In [ ]:
from pyDTDM import *
from pyDTDM.utils import *
from pyDTDM.BhuDM import PlateKinematicsParameters
import gplately
import geopandas as gpd
import pandas as pd
import os

import pyproj
# Point pyproj to the correct PROJ data directory
os.environ["PROJ_LIB"] = "<CONDA>/envs/EBMTest311/share/proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])

import yaml
try:
    from yaml import Cloader as Loader
except ImportError:

    from yaml import Loader

from plate_model_manager import PlateModelManager
import glob
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
def plotgdf(gdf,gplot,column=None,mollweide=False,time=0,cbar=False,quick=True,**kwargs):

    '''This function can be used to plot the reconstructed geodataframe at any time along with topologies and 
    features. If the data is large it will take a lot of time to plot. Turn quick to True to plot the data faster.
    However, there may be some issues with the colors.

    gdf: gpd.GeoDataFrame
    model: gplatey.PlateReconconstruction
    column: name of the colum to be plotted (str)
    time: reconstruction time (int)
    cbar: whether to display colorbar

    '''
    
    cmap = kwargs.get('cmap', None)
    vmin = kwargs.get('vmin', None)
    vmax = kwargs.get('vmax', None)
    label = kwargs.get('label', None)
    title=kwargs.get('title', None)
    features=kwargs.get('features',True)
    color=kwargs.get('color',None)
    markersize=kwargs.get('markersize',10)
    orientation=kwargs.get('orientation','vertical')
    shrink=kwargs.get('shrink',0.5)
    extend=kwargs.get('extend',None)
    
    central_longitude=kwargs.get('central_longitude',0)
    figsize=kwargs.get('figsize',(12,8))
    

    
    # fig = plt.figure(figsize=figsize, dpi=300)
    # gplot = gplately.PlotTopologies(model, coastlines=model.coastlines, continents=model.continents, time=time)

    ax = kwargs.get('ax', None)

    if ax is None:
        fig = plt.figure(figsize=figsize, dpi=300)
        if mollweide:
            ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=central_longitude))
            ax.gridlines(color='0.7',linestyle='--', xlocs=np.arange(-180,180,30), ylocs=np.arange(-90,90,30))
    
            mollweide_proj = f"+proj=moll +lon_0={central_longitude} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
            gdf=gdf.to_crs(mollweide_proj)
        else:
            ax = fig.add_subplot(111, projection=ccrs.PlateCarree(central_longitude=central_longitude))
            ax.gridlines(color='0.7',linestyle='--', xlocs=np.arange(-180,180,15), ylocs=np.arange(-90,90,15))
    else:
        fig = ax.figure

    # if mollweide:
    #     ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude = central_longitude))
    #     ax.gridlines(color='0.7',linestyle='--', xlocs=np.arange(-180,180,30), ylocs=np.arange(-90,90,30))
    
    #     mollweide_proj = f"+proj=moll +lon_0={central_longitude} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
    #     gdf=gdf.to_crs(mollweide_proj)
    # else:
    #     ax = fig.add_subplot(111, projection=ccrs.PlateCarree(central_longitude = central_longitude))
    #     ax.gridlines(color='0.7',linestyle='--', xlocs=np.arange(-180,180,15), ylocs=np.arange(-90,90,15))
    
        
        
    if features:
    
        # Plot shapefile features, subduction zones and MOR boundaries at time Ma
        gplot.time = time # Ma
        # gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
        gplot.plot_coastlines(ax, color='grey',alpha=0.2)
        # gplot.plot_ridges_and_transforms(ax, color='k')
        gplot.plot_trenches(ax, color='k',alpha=0.4)
        gplot.plot_subduction_teeth(ax, color='k',alpha=0.4)

        gplot.plot_ridges(ax, color='k',alpha=0.4)
        gplot.plot_transforms(ax, color='k',alpha=0.4)
        gplot.plot_misc_boundaries(ax, color='k',alpha=0.4)

        # Plot the GeoDataFrame
    
    if quick:
        da=df_to_NetCDF(x=gdf["Longitude"],y=gdf["Latitude"],z=gdf[column],grid_resolution=0.2,
                        lon_bin_edges=np.arange(-180,180,0.2),
                        lat_bin_edges=np.arange(-90,90,0.2))
        plot=gplot.plot_grid(ax=ax, grid=da,**{'cmap':cmap,'vmax':vmax,'vmin': vmin})
    else:
        plot = gdf.plot(ax=ax, cmap=cmap, column=column,vmax=vmax,vmin=vmin,color=color,markersize=markersize)
    

                                             # 'label':f'{column}'})
    if cbar:
        # Create a ScalarMappable object
        sm = cm.ScalarMappable(cmap=cmap)
        sm.set_array(gdf[column])
        sm.set_clim(vmin, vmax)
        
        # Add colorbar using the same Axes object used for plotting
        colorbar = plt.colorbar(sm, ax=ax, orientation=orientation,shrink=shrink,extend=extend, label=label)
        colorbar.set_label(label)
    
    ax.set_global()
    
    return ax  

In [ ]:
from plate_model_manager import PlateModelManager

print(PlateModelManager().get_available_model_names())

In [ ]:
pm_manager = PlateModelManager()
model = pm_manager.get_model("alfonso2024", data_dir="<DATA_ROOT>/Paper/Submission/SupplementaryMaterial/plate-model-repo")
# rotation_model = model.get_rotation_model()
# topology_features = model.get_topologies()
static_polygons = model.get_static_polygons()
coastlines = model.get_coastlines()
continents = model.get_continental_polygons()

In [ ]:
### topologies
# topology_filenames=glob.glob("<DATA_ROOT>/Raw/plate_model/*.gpml")
# rotation_filenames=glob.glob("<DATA_ROOT>/Raw/plate_model/*.rot")


In [ ]:
PK = PlateKinematicsParameters(
    model_name='alfonso2024',
    # topology_filenames,  # List of topology filenames
    # rotation_filenames,  # List of rotation filenames
    # static_polygons,     # Static polygons for features
    # # agegrid=agegrid,     # Age grid for the model
    # coastlines=coastlines,  # Coastline data
    # continents=continents,  # Continents data
    # anchor_plate_id=Mantle_ID  # ID for the mantle-optimized reference frame
    working_directory="<DATA_ROOT>/Paper/Submission/SupplementaryMaterial/plate-model-repo"
)


In [ ]:
time=0
# Create a plot for topologies using the PlateKinematicsParameters instance
gplot = gplately.PlotTopologies(PK.model, coastlines=coastlines, continents=continents, time=time)

In [ ]:
# grid_data=pd.read_csv("spatiotemporal_grid_predictions.csv") 
grid_data=pd.read_csv("<DATA_ROOT>/Paper/Zenodo_DataBundle/data/outputs/spatiotemporal_grid_predictions_latest.csv")
grid_gdf=gpd.GeoDataFrame(grid_data,geometry=gpd.points_from_xy(grid_data['lon'],grid_data['lat']),crs="EPSG:4326")

In [ ]:
grid_gdf["age (Ma)"].hist()

In [ ]:
copper_deposits=gpd.read_file("<DATA_ROOT>/Paper/Zenodo_DataBundle/data/deposits/STAMP_training_data_Alfonso2024.csv")

In [ ]:
copper_deposits=copper_deposits[copper_deposits['label']=='positive']

In [ ]:
copper_deposits.columns

In [ ]:
copper_deposits['tonnage_mt']=copper_deposits['tonnage_mt'].fillna(0.1)
copper_deposits['tonnage_mt'] = pd.to_numeric(copper_deposits['tonnage_mt'], errors='coerce')
copper_deposits['age (Ma)'] = pd.to_numeric(copper_deposits['age (Ma)'], errors='coerce')
copper_deposits['age (Ma)']

In [ ]:
copper_deposits_gdf=gpd.GeoDataFrame(copper_deposits,geometry=gpd.points_from_xy(copper_deposits['lon'],copper_deposits['lat']),crs="EPSG:4326")    
# copper_deposits_gdf.plot(markersize=sizes)
plt.show()

In [ ]:
print(len(copper_deposits['lon']))
print(len(copper_deposits['lat']))
print(len(copper_deposits['tonnage_mt']))


In [ ]:
grid_gdf["age (Ma)"]

In [ ]:
grid_gdf[grid_gdf["age (Ma)"] == 58]

In [ ]:
copper_deposits_gdf['tonnage_mt']=copper_deposits_gdf['tonnage_mt'].fillna(1.0)

In [ ]:
interval=5
time=62
import cmcrameri.cm as cmc
try:
    combined_probs_dfc=grid_gdf[grid_gdf['age (Ma)']==time].copy()
    copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
    copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+interval) & (copper_deposits_gdf['age (Ma)']>=time-interval)].copy()
    # Create subplots
    ncols = 1
    nrows = 1
    fig, ax = plt.subplots(nrows, ncols, 
                            figsize=(15, 3*nrows),
                            dpi=300, 
                            subplot_kw={'projection': ccrs.Orthographic(-90,20)})
    # Plot shapefile features, subduction zones and MOR boundaries at time Ma
    gplot.time = time # Ma
    gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
    gplot.plot_coastlines(ax, color='grey',alpha=0.7,lw=0.1)
    gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
   
        # The distribution of points in the velocity domain: set global extent with 5 degree intervals
    Xnodes = np.arange(-180,180,10)
    Ynodes = np.arange(-90,90,10)


    # Create a lat-lon mesh and convert to 1d lat-lon arrays
    # x, y = np.meshgrid(Xnodes,Ynodes)
    # x = x.flatten()
    # y = y.flatten()

    # vel_x, vel_y = PK.model.get_point_velocities(x, y, time, return_east_north_arrays=True)
    # vel_mag = np.hypot(vel_x, vel_y)

    # ax.streamplot(x, y, vel_x, vel_y, color=vel_mag, transform=ccrs.PlateCarree(), 
    #                 linewidth=0.005*vel_mag, cmap=plt.cm.turbo, density=1)
        # Get colorbar axis
    # gplot.plot_ridges_and_transforms(ax, color='k')

    # nc=xr.open_dataarray(f"<DATA_ROOT>/Raw/source_data/CarbonateThickness/uncompacted_carbonate_thickness_{time}Ma.nc")
    # nc.plot(ax=ax, cmap='cool',
    #          vmin=0, vmax=300, transform=ccrs.PlateCarree(), add_colorbar=False)
        
  
    sizes = np.sqrt(copper_deposits['tonnage_mt']) * 10
    # combined_probs_dfc.plot(ax=ax, column='Prospectivity Score', cmap='rainbow', markersize=0.1, 
    #                         # legend=True,
    #                         alpha=1.0,
    #                         legend_kwds={'label': 'Probability'},
    #                         transform=ccrs.PlateCarree())
    
    combined_probs_dfc.plot(
    ax=ax, 
    column='Prospectivity Score', 
    cmap=cmc.batlow, # Highly recommended 
    markersize=0.1, 
    alpha=1.0,
    vmin=0,
    vmax=1.0,
    legend=True,
    legend_kwds={'label': 'Spatiotemporal \n Prospectivity', 'orientation': 'vertical', 'shrink': 0.4,
               },
    transform=ccrs.PlateCarree()
)
    # copper_deposits_gdf.plot(markersize=sizes,transform=ccrs.PlateCarree())
    if not copper_deposits_gdfc.empty:
        
        # Drop bad rows explicitly
        copper_deposits_gdfc = copper_deposits_gdfc[
            copper_deposits_gdfc['tonnage_mt'].notna() &
            (copper_deposits_gdfc['tonnage_mt'] > 0) &
            copper_deposits_gdfc.geometry.notna()
        ]

        if len(copper_deposits_gdfc) > 0:
            ax.scatter(
                copper_deposits_gdfc.geometry.x,
                copper_deposits_gdfc.geometry.y,
                s=np.sqrt(copper_deposits_gdfc['tonnage_mt']) * 0.8,
                # color='cyan',
                # edgecolor='white',
                linewidth=0.5,
                    # color='#ffd966',      # muted gold
                color='red',  
                edgecolor='#3a3a3a',
                alpha=0.5,
                transform=ccrs.PlateCarree()
            )
                # Drop bad rows explicitly
        copper_deposits_gdfc = copper_deposits_gdfc[
            copper_deposits_gdfc['tonnage_mt'].notna() &
            (copper_deposits_gdfc['tonnage_mt'] > 0) &
            copper_deposits_gdfc.geometry.notna()
        ]
    
  
    gplot.plot_trenches(ax, color='k',alpha=0.5,lw=0.2)
    gplot.plot_subduction_teeth(ax, color='k',alpha=0.5,lw=0.1)

    gplot.plot_ridges(ax, color='black',alpha=0.7,lw=0.2)
    gplot.plot_transforms(ax, color='black',alpha=0.7,lw=0.2)
    gplot.plot_misc_boundaries(ax, color='black',alpha=0.7,lw=0.1)  
    gplot.plot_plate_motion_vectors(ax, color='black', alpha=0.3,width=0.005,spacingX=20, spacingY=20)



    
    cbar = ax.get_figure().axes[-1]

    # Set label and style
    cbar.set_ylabel(
        "Spatiotemporal\nProspectivity",
        fontsize=6,
        # rotation=90,
        labelpad=10
    )

    cbar.tick_params(labelsize=6)

        # if len(copper_deposits_gdfc) > 0:
        #     ax.scatter(
        #         copper_deposits_gdfc.geometry.x,
        #         copper_deposits_gdfc.geometry.y,
        #         s=copper_deposits_gdfc['weights'],
        #         # color='cyan',
        #         # edgecolor='white',
        #         linewidth=0.5,
        #          color='#ffd966',      # muted gold
        #         edgecolor='#3a3a3a',
        #         alpha=0.5,
        #         transform=ccrs.PlateCarree()
        #     )

    plt.title(f'{time} Ma', fontsize=16)
    ax.set_global()
    # plt.savefig(f"<DATA_ROOT>/Figure/spatiotemporal_plots/spatiotemporal_plot_{time}.png", dpi=300, bbox_inches='tight')
    # plt.close()
except Exception as e:
    print(f"Error processing time {time} Ma: {e}")
    # break

## Key-parameter multi-panel figure (deep-time mineralisation windows)

Multi-panel evolution figure: **rows = parameters** (crustal thickness, carbonate thickness,
spatiotemporal prospectivity), **columns = key time steps**, with **plate-motion vectors**,
closed plate topologies, and **deposits forming in each window** overlaid. Crustal & carbonate
thickness are the **continuous reconstructed rasters** (`source_data/.../*_{t}Ma.nc`);
prospectivity is the model's `grid_gdf`. Edit `TIMES` / `PARAMS` / colormaps in the config cell.

In [ ]:
# ====== KEY-PARAMETER MULTI-PANEL — EDIT HERE ===============================
import os, numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt, cartopy.crs as ccrs
import cmcrameri.cm as cmc

TIMES      = [62, 50, 27, 20]            # Ma — key mineralisation windows (edit freely)
INTERVAL   = 5                            # show deposits with |age - t| <= INTERVAL Ma
PROJ       = ccrs.Orthographic(-90, 20)   # closed topologies (your working setup)
EXTENT     = [-185, -90, 18, 90]         # Americas (N+S Cordillera); None = full globe
MARKERSIZE = 3.0                          # grid-point size (prospectivity)
SHOW_VECTORS, SHOW_DEPOSITS = True, True
OUTDIR     = "<DATA_ROOT>/Figure/keyfigures"

_SRC     = "<DATA_ROOT>/Raw/source_data"
CRUST_NC = _SRC + "/CrustalThickness/crustal_thickness_{t}Ma.nc"
CARB_NC  = _SRC + "/CarbonateThickness/uncompacted_carbonate_thickness_{t}Ma.nc"

# rows of the figure. kind="nc" -> continuous reconstructed raster (var z);
# kind="grid" -> column of grid_gdf (model output). continents_on_top=True for sea-floor
# fields (carbonate) so land reads grey and the oceanic signal shows; False for crust/prospectivity.
# Perceptually-uniform, colourblind-safe cmcrameri maps (high-impact-journal friendly).
PARAMS = [
    dict(kind="nc",   path=CRUST_NC, cmap=cmc.lajolla, vmin=30000, vmax=50000,
         label="Crustal thickness (m)",   continents_on_top=False),
    dict(kind="nc",   path=CARB_NC,  cmap=cmc.oslo_r,  vmin=0,     vmax=300,
         label="Carbonate thickness (m)", continents_on_top=True),
    dict(kind="grid", col="Prospectivity Score", cmap=cmc.batlow, vmin=0, vmax=1.0,
         label="Spatiotemporal\nprospectivity", continents_on_top=False),
]

In [ ]:
def key_panel(ax, time, p):
    """One parameter `p` at `time` Ma on a closed-topology orthographic globe."""
    gplot.time = time
    if not p.get("continents_on_top"):
        gplot.plot_continents(ax, facecolor="#dededa", alpha=0.55, lw=0, zorder=1)
    # --- parameter field ---
    if p["kind"] == "nc":
        nc = xr.open_dataarray(p["path"].format(t=time))
        xn = "lon" if "lon" in nc.coords else "x"
        yn = "lat" if "lat" in nc.coords else "y"
        sct = ax.pcolormesh(nc[xn], nc[yn], nc.values, cmap=p["cmap"],
                            vmin=p["vmin"], vmax=p["vmax"], transform=ccrs.PlateCarree(),
                            shading="auto", rasterized=True, zorder=2)
    else:
        g = grid_gdf[grid_gdf["age (Ma)"] == time]
        sct = ax.scatter(g.geometry.x, g.geometry.y,
                         c=pd.to_numeric(g[p["col"]], errors="coerce"),
                         cmap=p["cmap"], vmin=p["vmin"], vmax=p["vmax"],
                         s=MARKERSIZE, marker="s", linewidths=0,
                         transform=ccrs.PlateCarree(), rasterized=True, zorder=2)
    if p.get("continents_on_top"):
        gplot.plot_continents(ax, facecolor="#dededa", alpha=0.92, lw=0, zorder=3)
    gplot.plot_coastlines(ax, color="#9a948b", alpha=0.8, lw=0.15, zorder=4)
    # plate boundaries (closed topologies resolved at this time)
    gplot.plot_trenches(ax, color="k", lw=0.4, alpha=0.8, zorder=4)
    gplot.plot_subduction_teeth(ax, color="k", lw=0.1, alpha=0.7, zorder=4)
    gplot.plot_ridges(ax, color="#b8412e", lw=0.3, alpha=0.8, zorder=4)
    if SHOW_VECTORS:
        gplot.plot_plate_motion_vectors(ax, color="0.15", alpha=0.5,
                                        width=0.004, spacingX=15, spacingY=15, zorder=5)
    # deposits forming within +/- INTERVAL of this time (open rings: read on any cmap)
    if SHOW_DEPOSITS:
        d = copper_deposits_gdf
        d = d[(d["age (Ma)"] <= time + INTERVAL) & (d["age (Ma)"] >= time - INTERVAL)]
        d = d[d["tonnage_mt"].notna() & (d["tonnage_mt"] > 0) & d.geometry.notna()]
        if len(d):
            sz = np.sqrt(pd.to_numeric(d["tonnage_mt"])) * 1.8
            ax.scatter(d.geometry.x, d.geometry.y, s=sz*1.7, facecolor="none",
                       edgecolor="k", linewidth=0.9, alpha=0.55, transform=ccrs.PlateCarree(), zorder=6)
            ax.scatter(d.geometry.x, d.geometry.y, s=sz, facecolor="none",
                       edgecolor="white", linewidth=0.6, alpha=0.95, transform=ccrs.PlateCarree(), zorder=6)
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree()) if EXTENT else ax.set_global()
    return sct

In [ ]:
os.makedirs(OUTDIR, exist_ok=True)
nr, nc_ = len(PARAMS), len(TIMES)
fig, axes = plt.subplots(nr, nc_, figsize=(3.3*nc_, 3.15*nr), dpi=160,
                         subplot_kw={"projection": PROJ})
axes = np.atleast_2d(axes)
for j, t in enumerate(TIMES):
    axes[0, j].set_title(f"{t} Ma", fontsize=15, fontweight="bold", pad=4)
for i, p in enumerate(PARAMS):
    sct = None
    for j, t in enumerate(TIMES):
        sct = key_panel(axes[i, j], t, p)
    cb = fig.colorbar(sct, ax=list(axes[i, :]), location="right",
                      fraction=0.018, pad=0.012, shrink=0.9, extend="max")
    cb.set_label(p["label"], fontsize=10); cb.ax.tick_params(labelsize=8)
fig.suptitle("Deep-time evolution across key copper-mineralisation windows",
             fontsize=16, fontweight="bold", y=0.995)
fig.savefig(f"{OUTDIR}/key_parameters_multipanel.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUTDIR}/key_parameters_multipanel.svg", bbox_inches="tight")
print("saved:", f"{OUTDIR}/key_parameters_multipanel.png")

In [ ]:
time=50
import cmcrameri.cm as cmc

create_directory_if_not_exists("<DATA_ROOT>/Figure/spatiotemporal_plots/")
for time in range(170,-1,-1):
    try:
        combined_probs_dfc=grid_gdf[grid_gdf['age (Ma)']==time].copy()
        copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
        copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+2) & (copper_deposits_gdf['age (Ma)']>=time-2)].copy()
        # Create subplots
        ncols = 1
        nrows = 1
        fig, ax = plt.subplots(nrows, ncols, 
                                figsize=(15, 3*nrows),
                                dpi=300, 
                                subplot_kw={'projection': ccrs.Orthographic(-90,20)})
        # Plot shapefile features, subduction zones and MOR boundaries at time Ma
        gplot.time = time # Ma
        gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
        gplot.plot_coastlines(ax, color='grey',alpha=0.7,lw=0.1)
        gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
        # gplot.plot_ridges_and_transforms(ax, color='k')
        gplot.plot_trenches(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_subduction_teeth(ax, color='k',alpha=0.1,lw=0.1)

        gplot.plot_ridges(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_transforms(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_misc_boundaries(ax, color='k',alpha=0.1,lw=0.1)
        sizes = np.sqrt(copper_deposits['tonnage_mt']) * 10
        # combined_probs_dfc.plot(ax=ax, column='Prospectivity Score', cmap='rainbow', markersize=0.1, 
        #                         # legend=True,
        #                         alpha=1.0,
        #                         legend_kwds={'label': 'Probability'},
        #                         transform=ccrs.PlateCarree())
        
        combined_probs_dfc.plot(
        ax=ax, 
        column='Prospectivity Score', 
        cmap=cmc.batlow, # Highly recommended over rainbow
        markersize=0.1, 
        alpha=1.0,
        vmin=0,
        vmax=0.8,
        # legend=True,
        # legend_kwds={'label': 'Porphyry Prospectivity Probability', 'orientation': 'horizontal', 'shrink': 0.6},
        transform=ccrs.PlateCarree()
    )
        # copper_deposits_gdf.plot(markersize=sizes,transform=ccrs.PlateCarree())
        if not copper_deposits_gdfc.empty:
            
            # Drop bad rows explicitly
            copper_deposits_gdfc = copper_deposits_gdfc[
                copper_deposits_gdfc['tonnage_mt'].notna() &
                (copper_deposits_gdfc['tonnage_mt'] > 0) &
                copper_deposits_gdfc.geometry.notna()
            ]

            if len(copper_deposits_gdfc) > 0:
                ax.scatter(
                    copper_deposits_gdfc.geometry.x,
                    copper_deposits_gdfc.geometry.y,
                    s=np.sqrt(copper_deposits_gdfc['tonnage_mt']) * 0.8,
                    # color='cyan',
                    # edgecolor='white',
                    linewidth=0.5,
                     color='#ffd966',      # muted gold
                    edgecolor='#3a3a3a',
                    alpha=0.5,
                    transform=ccrs.PlateCarree()
                )
                  # Drop bad rows explicitly
            copper_deposits_gdfc = copper_deposits_gdfc[
                copper_deposits_gdfc['tonnage_mt'].notna() &
                (copper_deposits_gdfc['tonnage_mt'] > 0) &
                copper_deposits_gdfc.geometry.notna()
            ]

            # if len(copper_deposits_gdfc) > 0:
            #     ax.scatter(
            #         copper_deposits_gdfc.geometry.x,
            #         copper_deposits_gdfc.geometry.y,
            #         s=copper_deposits_gdfc['weights'],
            #         # color='cyan',
            #         # edgecolor='white',
            #         linewidth=0.5,
            #          color='#ffd966',      # muted gold
            #         edgecolor='#3a3a3a',
            #         alpha=0.5,
            #         transform=ccrs.PlateCarree()
            #     )

        plt.title(f'{time} Ma', fontsize=16)
        ax.set_global()
        # plt.savefig(f"<DATA_ROOT>/Figure/spatiotemporal_plots/spatiotemporal_plot_{time}.png", dpi=300, bbox_inches='tight')
        # plt.close()
    except Exception as e:
        print(f"Error processing time {time} Ma: {e}")
    break

In [ ]:
grid_gdf[
    'carbonate_thickness (m)'
].hist(bins=50)

In [ ]:
copper_deposits_gdfc.columns

In [ ]:
time=50
import cmcrameri.cm as cmc


image_folder= "<DATA_ROOT>/Figure/carbonate_spatiotemporal_plots/"
create_directory_if_not_exists(image_folder)
for time in range(170,-1,-5):
    try:
        combined_probs_dfc=grid_gdf[grid_gdf['age (Ma)']==time].copy()
        copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
        copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+2) & (copper_deposits_gdf['age (Ma)']>=time-2)].copy()
        # Create subplots
        ncols = 1
        nrows = 1
        fig, ax = plt.subplots(nrows, ncols, 
                                figsize=(15, 3*nrows),
                                dpi=300, 
                                subplot_kw={'projection': ccrs.Orthographic(-90,20)})
        # Plot shapefile features, subduction zones and MOR boundaries at time Ma
        gplot.time = time # Ma
        gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
        gplot.plot_coastlines(ax, color='grey',alpha=0.7,lw=0.1)
        gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
        # gplot.plot_ridges_and_transforms(ax, color='k')
        gplot.plot_trenches(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_subduction_teeth(ax, color='k',alpha=0.1,lw=0.1)

        gplot.plot_ridges(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_transforms(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_misc_boundaries(ax, color='k',alpha=0.1,lw=0.1)
        sizes = np.sqrt(copper_deposits['tonnage_mt']) * 10
        # combined_probs_dfc.plot(ax=ax, column='Prospectivity Score', cmap='rainbow', markersize=0.1, 
        #                         # legend=True,
        #                         alpha=1.0,
        #                         legend_kwds={'label': 'Probability'},
        #                         transform=ccrs.PlateCarree())
        
        # nc=df_to_NetCDF(x=combined_probs_dfc.geometry.x, y=combined_probs_dfc.geometry.y, z=combined_probs_dfc['carbonate_thickness (m)'], grid_resolution=0.5)
        nc=xr.open_dataarray(f"<DATA_ROOT>/Raw/source_data/CarbonateThickness/uncompacted_carbonate_thickness_{time}Ma.nc")
        nc.plot(ax=ax, cmap=cmc.lapaz, vmin=0, vmax=300, transform=ccrs.PlateCarree(), add_colorbar=False)
        
        
    #     combined_probs_dfc.plot(
    #     ax=ax, 
    #     column='carbonate_thickness (m)', 
    #     cmap=cmc.lapaz, # Highly recommended over rainbow
    #     markersize=0.1, 
    #     alpha=1.0,
    #     vmin=0,
    #     vmax=300,
    #     # legend=True,
    #     # legend_kwds={'label': 'Porphyry Prospectivity Probability', 'orientation': 'horizontal', 'shrink': 0.6},
    #     transform=ccrs.PlateCarree()
    # )
        # copper_deposits_gdf.plot(markersize=sizes,transform=ccrs.PlateCarree())
        if not copper_deposits_gdfc.empty:
            
            # Drop bad rows explicitly
            copper_deposits_gdfc = copper_deposits_gdfc[
                copper_deposits_gdfc['tonnage_mt'].notna() &
                (copper_deposits_gdfc['tonnage_mt'] > 0) &
                copper_deposits_gdfc.geometry.notna()
            ]

            if len(copper_deposits_gdfc) > 0:
                ax.scatter(
                    copper_deposits_gdfc.geometry.x,
                    copper_deposits_gdfc.geometry.y,
                    s=np.sqrt(copper_deposits_gdfc['tonnage_mt']) * 0.8,
                    # color='cyan',
                    # edgecolor='white',
                    linewidth=0.5,
                     color='#ffd966',      # muted gold
                    edgecolor='#3a3a3a',
                    alpha=0.5,
                    transform=ccrs.PlateCarree()
                )

        plt.title(f'{time} Ma', fontsize=16)
        ax.set_global()
        plt.savefig(f"{image_folder}/spatiotemporal_plot_{time}.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Error processing time {time} Ma: {e}")
    # break


import glob
from PIL import Image
# Define the folder and pattern for your image files
# image_folder = f"<DATA_ROOT>/Figure/spatiotemporal_plots"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Probs.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)

In [ ]:
time=50
import cmcrameri.cm as cmc


image_folder= "<DATA_ROOT>/Figure/crustal_spatiotemporal_plots/"
create_directory_if_not_exists(image_folder)
for time in range(60,-1,-1):
    try:
        combined_probs_dfc=grid_gdf[grid_gdf['age (Ma)']==time].copy()
        copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
        copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+2) & (copper_deposits_gdf['age (Ma)']>=time-2)].copy()
        # Create subplots
        ncols = 1
        nrows = 1
        fig, ax = plt.subplots(nrows, ncols, 
                                figsize=(15, 3*nrows),
                                dpi=300, 
                                subplot_kw={'projection': ccrs.Orthographic(-90,20)})
        # Plot shapefile features, subduction zones and MOR boundaries at time Ma
        gplot.time = time # Ma
        gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
        gplot.plot_coastlines(ax, color='grey',alpha=0.7,lw=0.1)
        gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
        # gplot.plot_ridges_and_transforms(ax, color='k')
        gplot.plot_trenches(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_subduction_teeth(ax, color='k',alpha=0.1,lw=0.1)

        gplot.plot_ridges(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_transforms(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_misc_boundaries(ax, color='k',alpha=0.1,lw=0.1)
        sizes = np.sqrt(copper_deposits['tonnage_mt']) * 10
        # combined_probs_dfc.plot(ax=ax, column='Prospectivity Score', cmap='rainbow', markersize=0.1, 
        #                         # legend=True,
        #                         alpha=1.0,
        #                         legend_kwds={'label': 'Probability'},
        #                         transform=ccrs.PlateCarree())
        nc=xr.open_dataarray(f"<DATA_ROOT>/Raw/source_data/CarbonateThickness/uncompacted_carbonate_thickness_{time}Ma.nc")
        im1= nc.plot(ax=ax, cmap=cmc.nuuk_r, vmin=0, vmax=300, transform=ccrs.PlateCarree(), add_colorbar=False)
        
        nc=df_to_NetCDF(x=combined_probs_dfc.geometry.x, y=combined_probs_dfc.geometry.y, z=combined_probs_dfc['crustal_thickness_mean (m)'], grid_resolution=0.5)
        im2= nc.plot(ax=ax, cmap=cmc.lipari, vmin=0, vmax=50000, transform=ccrs.PlateCarree(), add_colorbar=False)

        # fig = ax.get_figure()

        # cbar1 = fig.colorbar(im1, ax=ax, orientation='vertical', shrink=0.5, pad=0.02)
        # cbar1.set_label("Carbonate Thickness (m)", fontsize=9)
        # cbar1.ax.tick_params(labelsize=8)

        # cbar2 = fig.colorbar(im2, ax=ax, orientation='vertical', shrink=0.5, pad=0.08)
        # cbar2.set_label("Crustal Thickness (m)", fontsize=9)
        # cbar2.ax.tick_params(labelsize=8)
    #     nc.plot.contour(
    #     ax=ax,
    #     levels=10,
    #     colors='black',
    #     linewidths=0.5,
    #     transform=ccrs.PlateCarree()
    # )
        
        
    #     combined_probs_dfc.plot(
    #     ax=ax, 
    #     column='carbonate_thickness (m)', 
    #     cmap=cmc.lapaz, # Highly recommended over rainbow
    #     markersize=0.1, 
    #     alpha=1.0,
    #     vmin=0,
    #     vmax=300,
    #     # legend=True,
    #     # legend_kwds={'label': 'Porphyry Prospectivity Probability', 'orientation': 'horizontal', 'shrink': 0.6},
    #     transform=ccrs.PlateCarree()
    # )
        # copper_deposits_gdf.plot(markersize=sizes,transform=ccrs.PlateCarree())
        if not copper_deposits_gdfc.empty:
            
            # Drop bad rows explicitly
            copper_deposits_gdfc = copper_deposits_gdfc[
                copper_deposits_gdfc['tonnage_mt'].notna() &
                (copper_deposits_gdfc['tonnage_mt'] > 0) &
                copper_deposits_gdfc.geometry.notna()
            ]

            if len(copper_deposits_gdfc) > 0:
                ax.scatter(
                    copper_deposits_gdfc.geometry.x,
                    copper_deposits_gdfc.geometry.y,
                    s=np.sqrt(copper_deposits_gdfc['tonnage_mt']) * 0.8,
                    # color='cyan',
                    # edgecolor='white',
                    linewidth=0.5,
                     color='#ffd966',      # muted gold
                    edgecolor='#3a3a3a',
                    alpha=0.5,
                    transform=ccrs.PlateCarree()
                )

        plt.title(f'{time} Ma', fontsize=16)
        ax.set_global()
        # plt.savefig(f"{image_folder}/spatiotemporal_plot_{time}.png", dpi=300, bbox_inches='tight')
        # plt.close()
    except Exception as e:
        print(f"Error processing time {time} Ma: {e}")
    break


In [ ]:
time=50
import cmcrameri.cm as cmc


image_folder= "<DATA_ROOT>/Figure/crustal_spatiotemporal_plots/"
create_directory_if_not_exists(image_folder)
for time in range(170,-1,-1):
    try:
        combined_probs_dfc=grid_gdf[grid_gdf['age (Ma)']==time].copy()
        copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
        copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+2) & (copper_deposits_gdf['age (Ma)']>=time-2)].copy()
        # Create subplots
        ncols = 1
        nrows = 1
        fig, ax = plt.subplots(nrows, ncols, 
                                figsize=(15, 3*nrows),
                                dpi=300, 
                                subplot_kw={'projection': ccrs.Orthographic(-90,20)})
        # Plot shapefile features, subduction zones and MOR boundaries at time Ma
        gplot.time = time # Ma
        gplot.plot_continents(ax, facecolor='grey', alpha=0.2)
        gplot.plot_coastlines(ax, color='grey',alpha=0.7,lw=0.1)
        gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
        # gplot.plot_ridges_and_transforms(ax, color='k')
        gplot.plot_trenches(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_subduction_teeth(ax, color='k',alpha=0.1,lw=0.1)

        gplot.plot_ridges(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_transforms(ax, color='k',alpha=0.1,lw=0.1)
        gplot.plot_misc_boundaries(ax, color='k',alpha=0.1,lw=0.1)
        sizes = np.sqrt(copper_deposits['tonnage_mt']) * 10
        # combined_probs_dfc.plot(ax=ax, column='Prospectivity Score', cmap='rainbow', markersize=0.1, 
        #                         # legend=True,
        #                         alpha=1.0,
        #                         legend_kwds={'label': 'Probability'},
        #                         transform=ccrs.PlateCarree())
        nc=xr.open_dataarray(f"<DATA_ROOT>/Raw/source_data/CarbonateThickness/uncompacted_carbonate_thickness_{time}Ma.nc")
        nc.plot(ax=ax, cmap=cmc.lapaz, vmin=0, vmax=300, transform=ccrs.PlateCarree(), add_colorbar=False)
        
        nc=df_to_NetCDF(x=combined_probs_dfc.geometry.x, y=combined_probs_dfc.geometry.y, z=combined_probs_dfc['crustal_thickness_mean (m)'], grid_resolution=0.5)
        nc.plot(ax=ax, cmap=cmc.lipari, vmin=0, vmax=50000, transform=ccrs.PlateCarree(), add_colorbar=False)
        
        
    #     combined_probs_dfc.plot(
    #     ax=ax, 
    #     column='carbonate_thickness (m)', 
    #     cmap=cmc.lapaz, # Highly recommended over rainbow
    #     markersize=0.1, 
    #     alpha=1.0,
    #     vmin=0,
    #     vmax=300,
    #     # legend=True,
    #     # legend_kwds={'label': 'Porphyry Prospectivity Probability', 'orientation': 'horizontal', 'shrink': 0.6},
    #     transform=ccrs.PlateCarree()
    # )
        # copper_deposits_gdf.plot(markersize=sizes,transform=ccrs.PlateCarree())
        if not copper_deposits_gdfc.empty:
            
            # Drop bad rows explicitly
            copper_deposits_gdfc = copper_deposits_gdfc[
                copper_deposits_gdfc['tonnage_mt'].notna() &
                (copper_deposits_gdfc['tonnage_mt'] > 0) &
                copper_deposits_gdfc.geometry.notna()
            ]

            if len(copper_deposits_gdfc) > 0:
                ax.scatter(
                    copper_deposits_gdfc.geometry.x,
                    copper_deposits_gdfc.geometry.y,
                    s=np.sqrt(copper_deposits_gdfc['tonnage_mt']) * 0.8,
                    # color='cyan',
                    # edgecolor='white',
                    linewidth=0.5,
                     color='#ffd966',      # muted gold
                    edgecolor='#3a3a3a',
                    alpha=0.5,
                    transform=ccrs.PlateCarree()
                )

        plt.title(f'{time} Ma', fontsize=16)
        ax.set_global()
        plt.savefig(f"{image_folder}/spatiotemporal_plot_{time}.png", dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Error processing time {time} Ma: {e}")
    # break


import glob
from PIL import Image
# Define the folder and pattern for your image files
# image_folder = f"<DATA_ROOT>/Figure/spatiotemporal_plots"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Probs.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)

In [ ]:

import glob
from PIL import Image
# Define the folder and pattern for your image files
image_folder = f"<DATA_ROOT>/Figure/spatiotemporal_plots"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Probs.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import imageio
import os

# Directory to store frames
output_dir = "frames"
os.makedirs(output_dir, exist_ok=True)

# Time range (Ma)
time_range = range(170, -1, -5)  # from 170 to 0 Ma (inclusive)

# Create and save each frame
for time in time_range:
    combined_probs_dfc = grid_gdf[grid_gdf['Age_Ma'] == time]
    copper_deposits_gdf= copper_deposits_gdf.to_crs(epsg=4326)
    copper_deposits_gdfc=copper_deposits_gdf[(copper_deposits_gdf['age (Ma)']<=time+2) & (copper_deposits_gdf['age (Ma)']>=time-2)].copy()

    # Create subplot
    fig, ax = plt.subplots(figsize=(15, 3),
                           dpi=300,
                           subplot_kw={'projection': ccrs.Orthographic(-90, 20)})

    # Set time for gplot
    gplot.time = time

    # Plot coastlines, boundaries, etc.
    # gplot.plot_coastlines(ax, facecolor='grey', alpha=0.7, lw=0.1)
    gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
    gplot.plot_trenches(ax, color='k', alpha=0.5, lw=0.3)
    gplot.plot_subduction_teeth(ax, color='k', alpha=0.5, lw=0.1)
    gplot.plot_ridges(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_transforms(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_misc_boundaries(ax, color='k', alpha=0.1, lw=0.3)

    # Plot data points
    combined_probs_dfc.plot(ax=ax, column='Probability_1',
                            markersize=0.1,
                            legend=False, 
                            cmap='rainbow',
                            vmax=0.7,
                            vmin=0,
                            transform=ccrs.PlateCarree())
    ax.scatter(
    copper_deposits_gdfc['geometry'].x,copper_deposits_gdfc['geometry'].y,
    s=np.sqrt(copper_deposits_gdfc['tonnage_mt'])*0.5,
    color='k',
    alpha=0.5,
    transform=ccrs.PlateCarree()
)

    # Add title
    ax.set_title(f"{time} Ma", fontsize=8)
    ax.set_global()

    # Save frame
    frame_path = os.path.join(output_dir, f"frame_{time:03d}.png")
    plt.savefig(frame_path, bbox_inches='tight')
    # break
    plt.close(fig)

print("✅ All frames saved!")


In [ ]:

import glob
from PIL import Image
# Define the folder and pattern for your image files
image_folder = f"<PATH_TO>/MineralProspectivityAI/workflows/SpatioTemporalProspectivity/frames/"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Probs.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import imageio
import os

# Directory to store frames
output_dir = "frames_sediment_thickness"
os.makedirs(output_dir, exist_ok=True)

# Time range (Ma)
time_range = range(170, -1, -1)  # from 170 to 0 Ma (inclusive)

# Create and save each frame
for time in time_range:
    combined_probs_dfc = grid_gdf[grid_gdf['Age_Ma'] == time]

    # Create subplot
    fig, ax = plt.subplots(figsize=(15, 3),
                           dpi=300,
                           subplot_kw={'projection': ccrs.Orthographic(-90, 20)})

    # Set time for gplot
    gplot.time = time

    # Plot coastlines, boundaries, etc.
    # gplot.plot_coastlines(ax, facecolor='grey', alpha=0.7, lw=0.1)
    gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
    gplot.plot_trenches(ax, color='k', alpha=0.5, lw=0.3)
    gplot.plot_subduction_teeth(ax, color='k', alpha=0.5, lw=0.1)
    gplot.plot_ridges(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_transforms(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_misc_boundaries(ax, color='k', alpha=0.1, lw=0.3)


    # Plot data points
    combined_probs_dfc.plot(ax=ax, column='sediment_thickness (m)',
                            markersize=0.1,
                            legend=False, 
                            cmap='Wistia',
                            vmax=3000,
                            vmin=0,
                            transform=ccrs.PlateCarree())

    # Add title
    ax.set_title(f"{time} Ma", fontsize=8)
    ax.set_global()

    # Save frame
    frame_path = os.path.join(output_dir, f"frame_{time:03d}.png")
    plt.savefig(frame_path, bbox_inches='tight')
    # break
    plt.close(fig)

print("✅ All frames saved!")


In [ ]:
combined_probs_dfc = grid_gdf[grid_gdf['Age_Ma'] == time]

# Create subplot
fig, ax = plt.subplots(figsize=(15, 3),
                        dpi=300,
                        subplot_kw={'projection': ccrs.Orthographic(-90, 20)})

# Set time for gplot
gplot.time = time

# Plot coastlines, boundaries, etc.
# gplot.plot_coastlines(ax, facecolor='grey', alpha=0.7, lw=0.1)
gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
gplot.plot_trenches(ax, color='k', alpha=0.5, lw=0.3)
gplot.plot_subduction_teeth(ax, color='k', alpha=0.5, lw=0.1)
gplot.plot_ridges(ax, color='k', alpha=0.1, lw=0.3)
gplot.plot_transforms(ax, color='k', alpha=0.1, lw=0.3)
gplot.plot_misc_boundaries(ax, color='k', alpha=0.1, lw=0.3)


# Plot data points
combined_probs_dfc.plot(ax=ax, column='sediment_thickness (m)',
                        markersize=0.1,
                        cmap='Wistia',
                        vmax=3000,
                        vmin=0,
                        legend=True, 
                        legend_kwds={'label': 'Sediment Thickness (m)','extend': 'max'},
                        transform=ccrs.PlateCarree())

# Add title
ax.set_title(f"{time} Ma", fontsize=8)
ax.set_global()


In [ ]:

import glob
from PIL import Image
# Define the folder and pattern for your image files
image_folder = f"<PATH_TO>/MineralProspectivityAI/workflows/SpatioTemporalProspectivity/frames_sediment_thickness/"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Sediment_thickness.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import imageio
import os

# Directory to store frames
output_dir = "frames_trench_velocity"
os.makedirs(output_dir, exist_ok=True)

# Time range (Ma)
time_range = range(170, -1, -1)  # from 170 to 0 Ma (inclusive)

# Create and save each frame
for time in time_range:
    combined_probs_dfc = grid_gdf[grid_gdf['Age_Ma'] == time]

    # Create subplot
    fig, ax = plt.subplots(figsize=(15, 3),
                           dpi=300,
                           subplot_kw={'projection': ccrs.Orthographic(-90, 20)})

    # Set time for gplot
    gplot.time = time

    # Plot coastlines, boundaries, etc.
    # gplot.plot_coastlines(ax, facecolor='grey', alpha=0.7, lw=0.1)
    gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
    gplot.plot_trenches(ax, color='k', alpha=0.5, lw=0.3)
    gplot.plot_subduction_teeth(ax, color='k', alpha=0.5, lw=0.1)
    gplot.plot_ridges(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_transforms(ax, color='k', alpha=0.1, lw=0.3)
    gplot.plot_misc_boundaries(ax, color='k', alpha=0.1, lw=0.3)

    # Plot data points
    combined_probs_dfc.plot(ax=ax, column='trench_velocity (cm/yr)',
                            markersize=0.1,
                            legend=False, 
                            cmap='coolwarm',
                            vmax=5,
                            vmin=-5,
                            transform=ccrs.PlateCarree())

    # Add title
    ax.set_title(f"{time} Ma", fontsize=8)
    ax.set_global()

    # Save frame
    frame_path = os.path.join(output_dir, f"frame_{time:03d}.png")
    plt.savefig(frame_path, bbox_inches='tight')
    # break
    plt.close(fig)

print("✅ All frames saved!")


In [ ]:
combined_probs_dfc = grid_gdf[grid_gdf['Age_Ma'] == time]

# Create subplot
fig, ax = plt.subplots(figsize=(15, 3),
                        dpi=300,
                        subplot_kw={'projection': ccrs.Orthographic(-90, 20)})

# Set time for gplot
gplot.time = time

# Plot coastlines, boundaries, etc.
# gplot.plot_coastlines(ax, facecolor='grey', alpha=0.7, lw=0.1)
gplot.plot_continents(ax, facecolor='lightgrey', alpha=0.7, lw=0.1)
gplot.plot_trenches(ax, color='k', alpha=0.5, lw=0.3)
gplot.plot_subduction_teeth(ax, color='k', alpha=0.5, lw=0.1)
gplot.plot_ridges(ax, color='k', alpha=0.1, lw=0.3)
gplot.plot_transforms(ax, color='k', alpha=0.1, lw=0.3)
gplot.plot_misc_boundaries(ax, color='k', alpha=0.1, lw=0.3)

# Plot data points
combined_probs_dfc.plot(ax=ax, column='trench_velocity (cm/yr)',
                        markersize=0.1,
                        # legend=False, 
                        cmap='coolwarm',
                        vmax=5,
                        vmin=-5,
                        legend=True, 
                        legend_kwds={'label': 'Trench Velocity (cm/yr)','extend': 'both'},
                        transform=ccrs.PlateCarree())

# Add title
ax.set_title(f"{time} Ma", fontsize=8)
ax.set_global()

In [ ]:

import glob
from PIL import Image
# Define the folder and pattern for your image files
image_folder = f"<PATH_TO>/MineralProspectivityAI/workflows/SpatioTemporalProspectivity/frames_trench_velocity/"
image_pattern = os.path.join(image_folder, '*.png')

# # Load all the images matching the pattern
image_files = sorted(glob.glob(image_pattern), key=lambda x: float(os.path.basename(x).split('/')[0].split('.')[0].split('_')[-1]), reverse=True)

# Create a list of images
images = [Image.open(image) for image in image_files]

# Save the images as a GIF
output_path = os.path.join(image_folder, 'Trench_velocity.gif')
images[0].save(output_path, save_all=True, append_images=images[1:], optimize=False, duration=300, loop=0)

print("GIF created successfully at", output_path)